In [1]:
# @title 0) 저장소 클론·pip 설치 (로컬에서는 생략 가능)
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
WORKDIR = Path(os.environ.get("COLAB_REPO_DIR", "/content/colab"))
MARK = WORKDIR / "src" / "mindscopex_analysis" / "__init__.py"


def _run(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


if MARK.is_file():
    print(f"이미 클론됨 → git pull 만 실행: {WORKDIR}")
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=False)
else:
    WORKDIR.parent.mkdir(parents=True, exist_ok=True)
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    _run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)])

os.environ["MINDSCOPEX_ROOT"] = str(WORKDIR.resolve())
os.chdir(WORKDIR)
print("cwd =", os.getcwd())
print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"])

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
        "torch",
        "transformers",
        "accelerate",
        "pandas",
        "pyyaml",
        "tqdm",
    ]
)
print("pip install -e . 완료")


이미 클론됨 → git pull 만 실행: /content/colab
cwd = /content/colab
MINDSCOPEX_ROOT = /content/colab
pip install -e . 완료


# Qwen 직관·논리·CRT 추론 및 결과 저장

`configs/qwen_reasoning_tasks.yaml`에 문제와(선택) 자동 검증 힌트를 둡니다. `preset: qwen_reasoning_pair`와 `model_tags`(예: `non_reasoning`, `reasoning`)로 **비추론형·추론형** Qwen을 같은 문제에 차례로 돌릴 수 있습니다. 결과는 `outputs/` 아래 실행 시각별 폴더에 저장됩니다.


## 설정 로드


In [2]:
import os
import re
import sys
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# 프로젝트
_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
        raise FileNotFoundError(f"MINDSCOPEX_ROOT={env!r} 아래에 {_REPO_MARK.as_posix()} 가 없습니다.")
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError(
        "src/mindscopex_analysis 을 찾지 못했습니다. Colab에서는 clone 후 프로젝트 루트로 이동하세요."
    )


_SRC = _find_repo_root() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from mindscopex_analysis.notebook_presets import DEFAULT_EXPERIMENT_BASE, MODEL_PRESETS, PRESET_CHOICES
from mindscopex_analysis.notebook_utils import deep_merge, dtype_from_str, project_root_from_notebook

print("준비 완료.")


준비 완료.


In [3]:
ROOT = project_root_from_notebook(Path.cwd())
YAML_PATH = ROOT / "configs" / "qwen_reasoning_tasks.yaml"

ACTIVE_PRESET = "qwen_reasoning_pair"

yaml_data: dict = {}
if YAML_PATH.is_file():
    with YAML_PATH.open(encoding="utf-8") as f:
        yaml_data = yaml.safe_load(f) or {}
    if isinstance(yaml_data.get("preset"), str):
        ACTIVE_PRESET = yaml_data["preset"]
        if ACTIVE_PRESET not in MODEL_PRESETS:
            raise ValueError(f"알 수 없는 preset {ACTIVE_PRESET!r}. 가능: {PRESET_CHOICES}")

EXP = deep_merge(deepcopy(DEFAULT_EXPERIMENT_BASE), MODEL_PRESETS[ACTIVE_PRESET])
EXP = deep_merge(EXP, {k: v for k, v in yaml_data.items() if k != "preset"})

GEN = EXP.get("generation") or {}
OUT_CFG = EXP.get("output") or {}
MODELS = EXP["models"]
RUNTIME = EXP["runtime"]

DEVICE = RUNTIME.get("device") or ("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = dtype_from_str(str(RUNTIME.get("dtype", "float32")))
TRC = bool(RUNTIME.get("trust_remote_code", True))

MODEL_KEYS = list(MODELS.keys())
# 실행할 태그: model_tags > model_tag > [첫 번째 모델]
_mlist = EXP.get("model_tags")
_one = EXP.get("model_tag")
if _mlist:
    MODEL_TAGS_TO_RUN = list(_mlist)
elif _one:
    MODEL_TAGS_TO_RUN = [_one]
else:
    MODEL_TAGS_TO_RUN = [MODEL_KEYS[0]]

for t in MODEL_TAGS_TO_RUN:
    if t not in MODELS:
        raise ValueError(f"model_tags 에 {t!r} 가 없습니다. 가능: {MODEL_KEYS}")

# 문제 목록: structured `problems` 우선, 없으면 prompt_groups 펼치기
tasks: list[dict] = []
problems = EXP.get("problems")
if problems:
    for p in problems:
        pid = p.get("id") or f"{p.get('category', 'x')}_{len(tasks)}"
        tasks.append(
            {
                "id": pid,
                "category": p.get("category", "general"),
                "prompt": p["prompt"],
                "verification": p.get("verification") or {},
            }
        )
else:
    pgroups = EXP.get("prompt_groups") or {}
    for cat, plist in pgroups.items():
        for i, text in enumerate(plist):
            tasks.append(
                {
                    "id": f"{cat}_{i}",
                    "category": cat,
                    "prompt": text,
                    "verification": {},
                }
            )

if not tasks:
    raise ValueError("실행할 문제가 없습니다. configs/qwen_reasoning_tasks.yaml 의 problems 또는 prompt_groups 를 채우세요.")

SUBDIR = str(OUT_CFG.get("subdir", "qwen_reasoning_runs"))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
OUT_DIR = ROOT / "outputs" / SUBDIR / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"preset={ACTIVE_PRESET!r}  DEVICE={DEVICE}")
print("model_tags:", MODEL_TAGS_TO_RUN)
for t in MODEL_TAGS_TO_RUN:
    print(f"  - {t}: {MODELS[t]!r}")
print(f"문제 수: {len(tasks)}  →  결과 디렉터리: {OUT_DIR}")


preset='qwen_single_instruct'  model[qwen_instruct]='Qwen/Qwen2.5-0.5B-Instruct'  DEVICE=cuda
문제 수: 8  →  결과 디렉터리: /content/colab/outputs/qwen_reasoning_runs/20260419_055202_utc


## 모델 순차 로드 (VRAM)

다음 셀에서 `model_tags` 순서로 모델을 **하나씩** 로드했다가 문제 전부 생성 후 해제합니다. 두 모델을 동시에 올리지 않습니다.


In [4]:
max_length = int(RUNTIME.get("max_length", 2048))
print("토크나이저·모델은 다음 '추론·검증·저장' 셀에서 model_tags 마다 순차 로드됩니다.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

로드 완료: Qwen/Qwen2.5-0.5B-Instruct on cuda


## 추론·검증·저장


In [5]:
def _verify(text: str, spec: dict) -> tuple[str, str]:
    """(상태, 메시지) — pass / fail / skip"""
    if not spec:
        return "skip", "검증 규칙 없음"
    t = text.lower()
    any_kw = spec.get("expect_contains_any") or []
    if any_kw and not any(k.lower() in t for k in any_kw):
        return "fail", f"다음 중 하나도 없음: {any_kw}"
    all_kw = spec.get("expect_contains_all") or []
    if all_kw and not all(k.lower() in t for k in all_kw):
        return "fail", f"다음을 모두 포함하지 않음: {all_kw}"
    rx = spec.get("expect_regex")
    if rx:
        try:
            if not re.search(rx, text, re.DOTALL):
                return "fail", f"정규식 불일치: {rx!r}"
        except re.error as e:
            return "fail", f"정규식 오류: {e}"
    return "pass", "규칙 충족"


def _format_input(tok, system_prompt: str, user_prompt: str) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    if hasattr(tok, "apply_chat_template"):
        return tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    if system_prompt:
        return f"{system_prompt}\n\nUser: {user_prompt}\nAssistant:"
    return f"User: {user_prompt}\nAssistant:"


import gc

system_prompt = str(GEN.get("system_prompt") or "")
max_new_tokens = int(GEN.get("max_new_tokens", 512))
temperature = float(GEN.get("temperature", 0.7))
top_p = float(GEN.get("top_p", 0.9))
do_sample = bool(GEN.get("do_sample", True))
extra_gen = GEN.get("extra_generate_kwargs") or {}

rows: list[dict] = []

for tag in MODEL_TAGS_TO_RUN:
    mid = MODELS[tag]
    print(f"\n{'=' * 60}\n▶ 로드: {tag}  →  {mid}\n{'=' * 60}", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(mid, trust_remote_code=TRC)
    model = AutoModelForCausalLM.from_pretrained(
        mid,
        torch_dtype=DTYPE,
        trust_remote_code=TRC,
    ).to(DEVICE).eval()
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    for task in tqdm(tasks, desc=f"{tag}"):
        prompt_text = task["prompt"]
        fmt = _format_input(tokenizer, system_prompt, prompt_text)
        inputs = tokenizer(fmt, return_tensors="pt", truncation=True, max_length=max_length)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if do_sample:
            gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"] = top_p
        gen_kwargs.update(extra_gen)

        with torch.no_grad():
            out_ids = model.generate(**inputs, **gen_kwargs)

        new_tokens = out_ids[0, inputs["input_ids"].shape[1] :]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        status, vmsg = _verify(answer, task.get("verification") or {})

        rows.append(
            {
                "id": task["id"],
                "category": task["category"],
                "prompt": prompt_text,
                "answer": answer,
                "verify_status": status,
                "verify_detail": vmsg,
                "model_tag": tag,
                "model_id": mid,
            }
        )

    del model, tokenizer
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
try:
    from IPython.display import display
except ImportError:
    display = print
display(df)

snap_path = OUT_DIR / "config_snapshot.yaml"
with snap_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(EXP, f, allow_unicode=True, sort_keys=False)

df.to_csv(OUT_DIR / "results.csv", index=False, encoding="utf-8-sig")
df.to_json(OUT_DIR / "results.json", orient="records", force_ascii=False, indent=2)

model_summary = ", ".join(f"`{MODELS[t]}` ({t})" for t in MODEL_TAGS_TO_RUN)
md_lines = [
    f"# Qwen 추론 실행 ({RUN_ID})",
    "",
    f"- preset: `{ACTIVE_PRESET}`",
    f"- 모델: {model_summary}",
    "",
    "| id | model_tag | category | verify | prompt (요약) |",
    "|----|-----------|----------|--------|---------------|",
]
for _, r in df.iterrows():
    pshort = (r["prompt"][:50] + "…") if len(str(r["prompt"])) > 50 else r["prompt"]
    md_lines.append(
        f"| {r['id']} | {r['model_tag']} | {r['category']} | {r['verify_status']} | {pshort} |"
    )

md_lines.append("")
md_lines.append("## 상세")
md_lines.append("")
for _, r in df.iterrows():
    md_lines.append(f"### {r['model_tag']} — {r['id']}")
    md_lines.append("")
    md_lines.append(f"- 검증: **{r['verify_status']}** — {r['verify_detail']}")
    md_lines.append("")
    md_lines.append("**질문**")
    md_lines.append("")
    md_lines.append(str(r["prompt"]))
    md_lines.append("")
    md_lines.append("**생성**")
    md_lines.append("")
    md_lines.append(str(r["answer"]))
    md_lines.append("")

(OUT_DIR / "README.md").write_text("\n".join(md_lines), encoding="utf-8")
print(f"저장 완료: {OUT_DIR}")


generate:   0%|          | 0/8 [00:00<?, ?it/s]

,id,category,prompt,answer,verify_status,verify_detail,model_tag,model_id
0,intuitive_0,intuitive,하늘은 왜 파란색인가요?,"죄송합니다, 저는 인공지능으로서 실제 눈에 보이는 화질이나 편의점 상품 등이 없습니...",skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct
1,intuitive_1,intuitive,봄에는 어떤 꽃이 피나요?,봄에는 다양한 꽃들이 피어나는 것이 특징입니다. 일반적으로 봄에는:\n\n1. 꽃잎...,skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct
2,intuitive_2,intuitive,물은 높은 곳에서 낮은 곳으로 흐른다. 맞나요?,"네, 맞습니다. 물의 방향과 빗방울이 흐르는 주어진 상황에 따라 물은 높은 곳에서 ...",skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct
3,intuitive_3,intuitive,사과의 색깔은 보통 무엇인가요?,"사과의 색깔은 주로 빨간색이며, 일반적으로 초록 또는 노란색으로 보입니다. 특히, ...",skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct
4,logical_0,logical,"A는 B보다 크고, B는 C보다 큽니다. A와 C 중 어느 것이 더 큰가요? 단계별...","A가 C보다 크다는 점에서 먼저 확인해야 합니다. 따라서 A가 C보다 크다면, A는...",skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct
5,logical_1,logical,모든 고양이는 동물입니다. 일부 동물은 날 수 있습니다. 따라서 일부 고양이는 날 ...,"아니요, 일부 고양이는 날 수 없을 수도 있습니다. 고양이는 종종 동물의 식사 시간...",skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct
6,logical_2,logical,"5명이 원형 테이블에 앉아있습니다. A는 B 옆에, C는 D 맞은편에 앉아있습니다....",E가 원형 테이블의 중앙으로 앉아 있습니다.,skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct
7,logical_3,logical,"한 농부가 늑대, 양, 양배추를 강 건너편으로 옮기려 합니다. 보트에는 한 번에 하...","물론이죠, 저는 AI 어시스턴트라서 가장 기본적인 방식으로 접근할 수 없습니다. 하...",skip,검증 규칙 없음,qwen_instruct,Qwen/Qwen2.5-0.5B-Instruct


저장 완료: /content/colab/outputs/qwen_reasoning_runs/20260419_055202_utc
